# Get metadata on available ESGF (CMIP6) datasets and store in json files
In this notebook ```ESMValTool``` is used to query available ESGF nodes to find which available climate model runs contain the variables we are interested in. The ```search_esgf``` function does a call to online meta-data servers and returns all available climate datasets that match our specified criteria. This notebook is used to make a choice which climate model to use for the impact analyses. In notebook 0a this choice is hard-coded.

## Note on names
Confusingly, names like ```dataset``` and not uniquely defined. A ```Dataset``` object in EMSValTool is a combination of climate model, ensemble member, experiment, etc. However, the 'climate model' in a ```Dataset``` is identified using the key ```dataset```. This is confusing, but outside of my control. See [Juckes et al 2020](https://doi.org/10.5194/gmd-13-201-2020) for a detailed description of the jargon used in CMIP6.

## Note on downloaded data
This notebook only queries for meta-data and does not download any other data. The actual data sits on different servers which may at any time be offline. This function returns a list of data that exists, but it does not guarantee that the data is available at the current time.

## Note on eWaterCycle vs ESMValTool
In this notebook we use the search_esgf function we build on top of ESMValTool, but the actual call to the meta data server is made from the ESMValTool package that we have wrapped in eWaterCycle. None of this would be possible without the work of the ESMValTool team. 

In [1]:
import ewatercycle.esmvaltool.search
from esmvalcore.config import CFG

from rich import print
import json
import os
from pathlib import Path

In [2]:
# Parameters, these get changed when running on HPC
# country = "australia"
# region_id = "camelsaus_102101A"
settings_path = "settings.json"

In [3]:
# Parameters
settings_path = "regions/lichtenstein/lamah_2410/settings.json"


In [4]:
# Setting for ESMValTool to make sure the online esgf resources are always used and
# we don't rely on locally cached information.
CFG['search_esgf'] = 'always'

In [5]:
# Load settings
# Read from the JSON file
with open(settings_path, "r") as json_file:
    settings = json.load(json_file)

In [6]:
experiment_of_interest = settings['CMIP_info']['experiments']
project_of_interest = settings['CMIP_info']['project']
# frequency_of_interest = settings['CMIP_info']['frequency']
# variables_of_interest = settings['CMIP_info']["variables"]


In [7]:
# We query the ESGF databases for available datasets. This calls external servers that host ESGF
# metadata which may be down at any moment. This query can take a long time to complete (minutes 
# to half an hour easily)

# Check for Spider or SRC
home_check = Path.home()

if "ewater" in str(home_check):
    file_parameters_path = f"regions/{settings["country"]}/{settings["caravan_id"]}/available_climate_datasets.json"  # Spider
else:
    file_parameters_path = f"available_climate_datasets.json"  # SRC

# We check if we already ran this test
need_to_run = True

if os.path.exists(file_parameters_path):
    display(f"File already exists: {file_parameters_path}")
    need_to_run = False

if need_to_run:
    valid_datasets = ewatercycle.esmvaltool.search.search_esgf(
        experiment=experiment_of_interest,
        project = project_of_interest,
        # frequency = frequency_of_interest,
        # variables = variables_of_interest
        frequency = "day",
        variables = ['rsds','pr','tas']
    )

'File already exists: regions/lichtenstein/lamah_2410/available_climate_datasets.json'

In [8]:
if need_to_run:
    display(valid_datasets)

In [9]:
if need_to_run:
    # Convert sets to lists for JSON serialization
    serializable_valid_datasets = {key: list(value) for key, value in valid_datasets.items()}
    
    # Write to a JSON file
    with open(f"regions/{settings["country"]}/{settings["caravan_id"]}/available_climate_datasets.json", "w") as json_file:
        json.dump(serializable_valid_datasets, json_file, indent=4)

In [10]:
# Read from the JSON file
with open(f"regions/{settings["country"]}/{settings["caravan_id"]}/available_climate_datasets.json", "r") as json_file:
    loaded_dict = json.load(json_file)

# Convert lists back to sets
restored_valid_datasets = {key: set(value) for key, value in loaded_dict.items()}

display(f"Restored dictionary: {restored_valid_datasets}")

"Restored dictionary: {'EC-Earth3-ESM-1': {'r5i1p1f1'}, 'NorESM2-LM': {'r27i1p1f1', 'r7i1p1f2', 'r40i1p1f1', 'r37i1p1f1', 'r35i1p1f1', 'r42i1p1f1', 'r33i1p1f1', 'r5i1p1f1', 'r12i1p1f1', 'r22i1p1f1', 'r14i1p1f1', 'r6i1p1f1', 'r25i1p1f1', 'r39i1p1f1', 'r26i1p1f1', 'r11i1p1f1', 'r30i1p1f1', 'r4i1p1f2', 'r8i1p1f1', 'r19i1p1f1', 'r20i1p1f1', 'r1i1p1f2', 'r16i1p1f1', 'r43i1p1f1', 'r32i1p1f1', 'r18i1p1f1', 'r13i1p1f1', 'r24i1p1f1', 'r7i1p1f1', 'r21i1p1f1', 'r29i1p1f1', 'r3i1p1f2', 'r2i1p1f2', 'r6i1p1f2', 'r10i1p1f2', 'r17i1p1f1', 'r34i1p1f1', 'r15i1p1f1', 'r4i1p1f1', 'r9i1p1f1', 'r2i1p1f1', 'r28i1p1f1', 'r23i1p1f1', 'r38i1p1f1', 'r10i1p1f1', 'r9i1p1f2', 'r41i1p1f1', 'r5i1p1f2', 'r1i1p1f1', 'r31i1p1f1', 'r8i1p1f2', 'r3i1p1f1', 'r36i1p1f1'}, 'CNRM-CM6-1-HR': {'r1i1p1f2'}, 'EC-Earth3-Veg': {'r6i1p1f1', 'r10i1p1f1', 'r11i1p1f1', 'r1i1p1f1', 'r5i1p1f1', 'r3i1p1f1', 'r13i1p1f1', 'r4i1p1f1', 'r2i1p1f1'}, 'CESM2-WACCM-FV2': {'r1i1p1f1'}, 'CNRM-CM6-1': {'r7i1p1f2', 'r23i1p1f2', 'r27i1p1f2', 'r24i1p1f2

In [11]:
# Adjust ensembles if needed, meaning we do not use an ensemble which is unavailable
pre_configured_ensembles = settings["CMIP_info"]["ensembles"]
pre_configured_dataset = settings["CMIP_info"]["dataset"][0]

available_ensembles = restored_valid_datasets.get(pre_configured_dataset)

final_ensembles = []
used = set()

for ensemble in pre_configured_ensembles:
    if ensemble in available_ensembles and ensemble not in used:
        final_ensembles.append(ensemble)
        used.add(ensemble)
    else:
        # Find a replacement that is valid and unused
        replacements = available_ensembles - used
        if not replacements:
            raise ValueError(
                f"No remaining valid ensembles for dataset '{pre_configured_dataset}'"
            )
        replacement = sorted(replacements)[0]
        final_ensembles.append(replacement)
        used.add(replacement)

settings["CMIP_info"]["ensembles"] = final_ensembles

# Write to a JSON file
with open(f"regions/{settings["country"]}/{settings["caravan_id"]}/settings.json", "w") as json_file:
    json.dump(settings, json_file, indent=4)